# 01 · Data overview & exploratory analysis

> **Public copy (NDA).** Outputs are cleared and private values are removed from the text. Run it locally with the NDA data in `data/`; never push it with outputs. Figures are not saved from this copy (`save` is a no-op): the public figures come from `public/reports/make_public_figures.py`.


**Schneider Electric × X-HEC Data Challenge 2026**: forecast raw-material (RM) prices 3, 6 and 9 months ahead for two Ring Main Units:

- **RM AirSeT**, made in Hungary
- **RM6 NE-IQI**, made in France

The forecasts then feed a sourcing and business recommendation.

This notebook covers six steps:

1. inventory every file
2. audit data quality
3. explore the price dynamics
4. test whether the steel-index workbook can fill the licensed-data gaps
5. quantify BOM cost exposure
6. set naive forecasting baselines for later models to beat

The written synthesis lives in [`DATA_OVERVIEW.md`](../DATA_OVERVIEW.md).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL
import src.data_loading as dl

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# --- chart system: validated categorical palette (fixed order), recessive chrome
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.prop_cycle": mpl.cycler(color=PAL), "lines.linewidth": 1.6, "legend.frameon": False,
    "legend.fontsize": 8, "font.family": "sans-serif", "figure.dpi": 110,
})
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
def save(fig, name): pass  # public copy: never write figures

dl.extract_archives()
rm, en, bom = dl.load_rm_wide(), dl.load_energy_wide(), dl.load_bom()
cat = dl.load_catalogs()
NAMES = dict(zip(cat["rm"].series_id, cat["rm"].material + " · " + cat["rm"].region))
NAMES.update(dict(zip(cat["energy"].series_id, cat["energy"].iloc[:, 1].astype(str) + " · " + cat["energy"].iloc[:, 2].astype(str))))
print("RM:", rm.shape, rm.index.min().date(), "→", rm.index.max().date(), "| Energy:", en.shape)

## 1 · File inventory
Every file and sheet, with shape and readability. Zips are extracted to `data/raw/`, which is gitignored.

In [ ]:
inv = dl.file_inventory()
summary = (inv.assign(group=inv.file.str.replace(r"^raw/Supplier_balancesheet/.*", "raw/Supplier_balancesheet/*.xlsx (9 files)", regex=True))
              .groupby("group").agg(sheets=("sheet", "nunique"), max_rows=("rows", "max"), size_kb=("size_kb", "max"), status=("status", "first")))
summary

In [ ]:
# Two copies of the RM master workbook disagree on the BOM
root_bom = pd.read_excel(dl.DATA / "RM_Prices_XHEC_Challenge.xlsx", sheet_name="BOM_Materials")
cmp = (bom[["product_ref", "material_family", "share_pct", "mass_kg"]].iloc[:14]
       .merge(root_bom[["product_ref", "material_family", "share_pct", "mass_kg"]].iloc[:14],
              on=["product_ref", "material_family"], suffixes=("_zip", "_root")))
cmp["mass_diff_kg"] = cmp.mass_kg_root - cmp.mass_kg_zip
print("data/RM_Prices_XHEC_Challenge.xlsx (root) has only the BOM sheet; values differ from data.zip ->")
cmp.dropna(subset=["mass_kg_zip"])

## 2 · Data quality
### 2.1 Missingness & publication lags

In [ ]:
both = pd.concat([rm, en], axis=1)
filled = both.loc[:, both.notna().any()]
fig, ax = plt.subplots(figsize=(11, 6.5))
ax.imshow(filled.notna().T.values, aspect="auto", cmap=mpl.colors.ListedColormap(["#f0efec", "#2a78d6"]), interpolation="nearest")
ax.set_yticks(range(filled.shape[1])); ax.set_yticklabels(filled.columns, fontsize=6.5)
xt = range(0, len(filled), 12); ax.set_xticks(list(xt)); ax.set_xticklabels([filled.index[i].strftime("%Y-%m") for i in xt])
ax.grid(False); ax.set_title("Observed (blue) vs missing (grey) months: RM + energy series")
save(fig, "01_missingness"); plt.show()

empty = both.columns[both.isna().all()].tolist()
last = filled.apply(lambda s: s.last_valid_index()).rename("last_obs")
gaps = pd.DataFrame({"n_obs": filled.notna().sum(), "first_obs": filled.apply(lambda s: s.first_valid_index()), "last_obs": last,
                     "internal_gaps": filled.apply(lambda s: s.loc[s.first_valid_index():s.last_valid_index()].isna().sum())})
print("Fully empty (licensed) columns:", empty)
print(gaps.last_obs.value_counts().sort_index())
gaps[gaps.internal_gaps.gt(0) | gaps.last_obs.lt("2026-09-01")]

### 2.2 Cross-source consistency: challenge LME-in-EUR (D01/D02) vs the stand-alone LME workbook

In [ ]:
lme = dl.load_lme()
chk = pd.DataFrame({"D01_challenge": rm.D01, "Cu_cash_EUR_lme": lme["Cu cash EUR/t"],
                    "D02_challenge": rm.D02, "Al_cash_EUR_lme": lme["Al cash EUR/t"]}).dropna()
chk["Cu_gap_%"] = 100 * (chk.D01_challenge / chk.Cu_cash_EUR_lme - 1)
chk["Al_gap_%"] = 100 * (chk.D02_challenge / chk.Al_cash_EUR_lme - 1)
print(chk[["Cu_gap_%", "Al_gap_%"]].describe().T[["mean", "std", "min", "max"]])
print("LME workbook extends to", lme.index.max().date(), "| status of last rows:", lme["Period status"].tail(2).tolist())

## 3 · Raw-material price dynamics
All series are rebased to **Nov-2016 = 100**, so materials in different units share one axis.

In [ ]:
key = ["D01", "D02", "S03", "S04", "S05", "S07", "S08", "S09", "S10", "S11", "S12", "S13"]
idx = rm[key].div(rm[key].bfill().iloc[0]).mul(100)
fig, axes = plt.subplots(3, 4, figsize=(13, 7.5), sharex=True, sharey=True)
for ax, c in zip(axes.flat, key):
    ax.axvspan(pd.Timestamp("2021-01-01"), pd.Timestamp("2022-12-01"), color="#f0efec", lw=0)
    ax.plot(idx.index, idx[c], color=PAL[0], lw=1.4); ax.axhline(100, color=AXIS, lw=0.8)
    ax.set_title(f"{c} · {NAMES.get(c, c)}"[:38], fontsize=8.5)
fig.suptitle("Raw-material prices, indexed Nov-2016 = 100 (shaded: 2021-22 commodity/energy shock)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "02_rm_indexed"); plt.show()

In [ ]:
lr = np.log(rm[key]).diff()
stats = pd.DataFrame({
    "last": rm[key].apply(lambda s: s.dropna().iloc[-1]),
    "YoY_%": 100 * (rm[key].ffill().iloc[-1] / rm[key].ffill().iloc[-13] - 1),
    "since_2016_%": 100 * (rm[key].ffill().iloc[-1] / rm[key].bfill().iloc[0] - 1),
    "ann_vol_%": 100 * lr.std() * np.sqrt(12),
    "vol_last12m_%": 100 * lr.iloc[-12:].std() * np.sqrt(12),
    "max_drawdown_%": 100 * (rm[key] / rm[key].cummax() - 1).min(),
    "peak_month": rm[key].idxmax().dt.strftime("%Y-%m"),
})
stats.index = [f"{c} {NAMES.get(c,'')}" for c in stats.index]
stats.sort_values("ann_vol_%", ascending=False)

In [ ]:
vol = (lr.rolling(12).std() * np.sqrt(12) * 100)
fig, ax = plt.subplots(figsize=(11, 3.8))
for i, c in enumerate(["D01", "D02", "S03", "S04", "S09"]):
    ax.plot(vol.index, vol[c], color=PAL[i], label=f"{c} {NAMES[c]}")
    ax.annotate(c, (vol.index[-1], vol[c].dropna().iloc[-1]), xytext=(4, 0), textcoords="offset points", fontsize=7.5, color=INK2, va="center")
ax.set_title("Rolling 12-month annualised volatility of log-returns (%)"); ax.legend(ncol=5, loc="upper left")
save(fig, "03_rolling_vol"); plt.show()

## 4 · Stationarity & seasonality
These tests set the modelling choices: whether to work in levels or differences, and whether a seasonal term is needed.

In [ ]:
rows = []
for c in key:
    s = np.log(rm[c].dropna())
    p_lvl = adfuller(s, autolag="AIC")[1]; p_dif = adfuller(s.diff().dropna(), autolag="AIC")[1]
    stl = STL(s, period=12, robust=True).fit()
    f_seas = max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.seasonal))
    f_trend = max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.trend))
    ac = s.diff().dropna().autocorr(1)
    rows.append((c, NAMES[c], p_lvl, p_dif, f_trend, f_seas, ac))
st = pd.DataFrame(rows, columns=["id", "series", "ADF_p_level", "ADF_p_dlog", "trend_strength", "seasonal_strength", "AC1_dlog"]).set_index("id")
st

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
c = "S09"; s = np.log(rm[c].dropna()); stl = STL(s, period=12, robust=True).fit()
axes[0].plot(s.index, np.exp(s), color=PAL[0], label="observed"); axes[0].plot(s.index, np.exp(stl.trend), color=PAL[1], label="STL trend")
axes[0].set_title(f"{c} {NAMES[c]}: level vs trend"); axes[0].legend()
axes[1].bar(st.index, st.AC1_dlog, color=PAL[0], width=0.6); axes[1].axhline(0, color=AXIS, lw=0.8)
axes[1].set_title("Lag-1 autocorrelation of monthly log-changes (momentum)")
fig.tight_layout(); save(fig, "04_momentum"); plt.show()

## 5 · Co-movement & energy as a leading indicator
### 5.1 Correlation of monthly log-returns between raw materials

In [ ]:
corr = lr.corr()
fig, ax = plt.subplots(figsize=(7.5, 6.2))
im = ax.imshow(corr, cmap=mpl.colors.LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"]), vmin=-1, vmax=1)
ax.set_xticks(range(len(key))); ax.set_xticklabels(key, rotation=90); ax.set_yticks(range(len(key))); ax.set_yticklabels(key); ax.grid(False)
for i in range(len(key)):
    for j in range(len(key)):
        ax.text(j, i, f"{corr.iat[i,j]:.2f}", ha="center", va="center", fontsize=6.5, color=INK)
fig.colorbar(im, ax=ax, shrink=0.75); ax.set_title("Correlation of monthly log-returns")
save(fig, "05_corr_returns"); plt.show()

### 5.2 Lead-lag: does energy move before metals and steel?
Cross-correlation corr(Δlog energy at *t−k*, Δlog material at *t*) for k = 0…9 months. A peak at k > 0 means energy leads the material.

In [ ]:
drivers = {"ED02": "TTF gas EUR/MWh", "ED01": "Brent EUR/bbl", "E09": "Power DE", "E13": "PPI energy EU", "E22": "IMF energy idx"}
targets = ["S09", "S10", "S08", "D02", "S11", "S04"]
le = np.log(en[list(drivers)].clip(lower=1e-6)).diff()
res = {}
for d in drivers:
    for t in targets:
        res[(d, t)] = [le[d].shift(k).corr(lr[t]) for k in range(10)]
ccf = pd.DataFrame(res, index=pd.Index(range(10), name="lag_k")).T
best = pd.DataFrame({"best_lag": ccf.abs().idxmax(axis=1), "corr_at_best": [ccf.loc[i, ccf.abs().loc[i].idxmax()] for i in ccf.index],
                     "corr_lag0": ccf[0]})
best.unstack(1)["best_lag"].join(best.unstack(1)["corr_at_best"], lsuffix="_lag", rsuffix="_r").round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, t in zip(axes, ["S09", "S08", "S11"]):
    for i, d in enumerate(["ED02", "ED01", "E09"]):
        ax.plot(range(10), ccf.loc[(d, t)], marker="o", ms=4, color=PAL[i], label=f"{d} {drivers[d]}")
    ax.axhline(0, color=AXIS, lw=0.8); ax.set_title(f"→ {t} {NAMES[t]}"[:40]); ax.set_xlabel("energy lead k (months)")
axes[0].legend(); fig.suptitle("Cross-correlation of monthly log-changes: energy(t−k) vs material(t)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); save(fig, "06_energy_ccf"); plt.show()

## 6 · Closing the licensed-data gap: MEPS and CRU vs the free proxies
The challenge dataset leaves **L01 (CR coil), L02 (304L stainless) and L03 (HDG)** empty because those series are licensed.
`Copia de Europe Steel INDEXES.xlsx` contains exactly these market prices in EUR/t, by country.
The questions are how well the free proxies (S09, S10 PPIs; S03 nickel) track them, and with what lag.

In [ ]:
meps, cru, ss = dl.load_meps(), dl.load_cru_flat(), dl.load_cru_stainless()
pick = lambda df, **kw: df.loc[np.logical_and.reduce([df[k] == v for k, v in kw.items()])].groupby("month").value.mean()
steel = pd.DataFrame({
    "CRU_CR_DE": pick(cru, country="Germany", product="CR", market="Domestic"),
    "CRU_HDG_DE": pick(cru, country="Germany", product="HDG", market="Domestic"),
    "CRU_HR_DE": pick(cru, country="Germany", product="HR", market="Domestic"),
    "CRU_CR_FR": pick(cru, country="France", product="CR", market="Domestic"),
    "MEPS_CR_DE": pick(meps, country="Germany", product="CR", unit="EUR"),
    "MEPS_HDG_DE": pick(meps, country="Germany", product="HDG", unit="EUR"),
    "CRU_SS304_DE": pick(ss, country="Germany", product="SS304"),
    "CRU_SS304_DE_alloy": ss.query("country=='Germany' and product=='SS304'").groupby("month").alloy.mean(),
})
steel.loc["2016-11":].describe().T[["count", "mean", "min", "max"]]

In [ ]:
pairs = [("CRU_CR_DE", "S09"), ("CRU_HDG_DE", "S10"), ("MEPS_CR_DE", "CRU_CR_DE"), ("CRU_SS304_DE", "S09"),
         ("CRU_SS304_DE_alloy", "S03"), ("CRU_CR_DE", "S04")]
J = steel.join(rm, how="inner").loc["2016-11":]
rows = []
for a, b in pairs:
    x, y = np.log(J[a]), np.log(J[b])
    lag_corr = {k: x.diff().corr(y.diff().shift(-k)) for k in range(-6, 7)}  # k>0: a leads b
    k_best = max(lag_corr, key=lambda k: abs(lag_corr[k]))
    rows.append((a, b, x.corr(y), x.diff().corr(y.diff()), k_best, lag_corr[k_best]))
gap = pd.DataFrame(rows, columns=["market_price", "proxy", "corr_levels(log)", "corr_dlog_lag0", "best_lag(+ = market leads)", "corr_at_best"])
gap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
b = J.loc["2016-11":]
for i, c in enumerate(["CRU_CR_DE", "MEPS_CR_DE", "CRU_HDG_DE", "S09", "S10"]):
    s = b[c] / b[c].dropna().iloc[0] * 100
    ax.plot(s.index, s, color=PAL[i], lw=1.4 if c.startswith(("CRU", "MEPS")) else 1.2, ls="-" if c.startswith(("CRU", "MEPS")) else "--", label=c)
ax.set_title("Carbon & galvanised steel: market prices (solid) vs Eurostat PPI proxies (dashed), Nov-2016 = 100"); ax.legend(ncol=5)
save(fig, "07_steel_market_vs_proxy"); plt.show()

In [ ]:
mp = dl.load_steel_mapping()
print("Reference→index mapping:", len(mp), "part references across", mp["Plant Name"].nunique(), "plants")
print("Lead time (months) used in price clauses:", mp["Lead Time (Months)"].value_counts().to_dict())
mp["Index"].value_counts().rename("n_references").to_frame().T

## 7 · BOM cost exposure: which materials drive the products' material cost?
Each BOM mass (`data.zip` version) is valued at the best EUR/t market series available. The proxies are listed below.

| Material | Price series |
|---|---|
| Carbon steel (CR) | CRU CR coil DE |
| HDG | CRU HDG DE |
| Stainless 304L | CRU SS304 DE, base + alloy surcharge |
| Copper | D01 |
| Aluminium | D02 |

Plastics have no EUR/t series. They are valued at **an assumed 2 500 EUR/t in 2021**, moved with the S11 PPI.

**Assumption:** the RM6 carbon-steel mass is missing in the BOM; it is labelled "to confirm".
It is estimated as an assumed share × the implied total mass (from the stainless mass and share). The latest BOM (v1) now gives it directly.

In [ ]:
plast = 2500 * rm.S11 / rm.S11.loc["2021"].mean()
price = pd.DataFrame({"Carbon steel": steel.CRU_CR_DE, "Galvanised steel": steel.CRU_HDG_DE,
                      "Stainless steel": steel.CRU_SS304_DE, "Copper": rm.D01, "Aluminium": rm.D02,
                      "Engineering plastic": plast, "Plastic": plast}).loc["2016-11":"2026-10"]
b = bom.dropna(subset=["product_ref"]).iloc[:14].copy()
rm6_total = b.query("product_ref=='RM6_NE_IQI_433520' and material_family=='Stainless steel'").eval("mass_kg/share_pct").iat[0]
b.loc[b.material_family == "Carbon steel (to confirm)", ["material_family", "mass_kg"]] = ["Carbon steel", round(0.45 * rm6_total, 1)]
b = b.dropna(subset=["mass_kg"]).query("material_family in @price.columns")
mass = b.groupby(["product_ref", "material_family"]).mass_kg.sum()
print(f"RM6 implied total mass ≈ {rm6_total:.0f} kg → carbon steel assumed {0.45*rm6_total:.0f} kg")
mass.unstack(0)

In [ ]:
costs = {}
for prod in mass.index.levels[0]:
    m = mass.loc[prod]
    costs[prod] = price[m.index].ffill().mul(m / 1000)  # EUR per unit
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=False)
summ = []
for ax, (prod, cdf) in zip(axes, costs.items()):
    order = cdf.mean().sort_values(ascending=False).index
    MAT_COLOR = {"Copper": PAL[0], "Stainless steel": PAL[1], "Carbon steel": PAL[2], "Engineering plastic": PAL[3],
                 "Plastic": PAL[3], "Galvanised steel": PAL[6], "Aluminium": PAL[5]}  # colour follows the material in both panels
    ax.stackplot(cdf.index, *[cdf[c] for c in order], labels=order, colors=[MAT_COLOR[c] for c in order], edgecolor="#fcfcfb", linewidth=0.4)
    ax.set_title(f"{prod}: priced material cost (EUR/unit)"); ax.legend(loc="upper left", ncol=2)
    tot = cdf.sum(axis=1)
    var_share = (cdf.diff().cov().sum() / tot.diff().var()).rename("share_of_cost_variance")
    summ.append(pd.DataFrame({"product": prod, "cost_last_EUR": cdf.iloc[-1], "share_of_cost_last": cdf.iloc[-1] / tot.iloc[-1],
                              "share_of_cost_variance": var_share}))
    print(f"{prod}: priced material cost {tot.loc[pd.Timestamp('2020-06-01')]:.0f} € (Jun-20) → peak {tot.max():.0f} € ({tot.idxmax():%b-%y}) → {tot.iloc[-1]:.0f} € (last)")
fig.tight_layout(); save(fig, "08_bom_cost"); plt.show()
pd.concat(summ).reset_index(names="material").set_index(["product", "material"]).sort_index()

## 8 · Competitive should-cost benchmark & epoxy

In [ ]:
bench = dl.load_benchmark()
comp = ["material", "overhead", "labour", "process"]
fig, ax = plt.subplots(figsize=(10, 3.6))
left = np.zeros(len(bench))
labels = [i.split(" · ")[0][:22] for i in bench.index]
for i, c in enumerate(comp):
    ax.barh(labels, bench[c], left=left, color=PAL[i], label=c, height=0.6, edgecolor="#fcfcfb", linewidth=1.5)
    left += bench[c].fillna(0).values
for y, v in enumerate(bench.total):
    ax.text(v + 30, y, f"€{v:,.0f}", va="center", fontsize=8, color=INK2)
ax.invert_yaxis(); ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.legend(ncol=4, loc="lower right")
ax.set_title("Teardown should-cost by competitor (EUR, 3-function RMU)")
save(fig, "09_benchmark"); plt.show()
bench[["role", "total", "material", "overhead", "Mass (g)", "List price", "Made in", "Insulating gas"]].assign(
    material_share=lambda d: d.material / d.total, eur_per_kg=lambda d: d.total / (d["Mass (g)"] / 1000))

### 8b · Selling-price benchmark (RMU IQI EU, synthetic)
`RMU_IQI_EU_Price_Benchmark_2017-2026_Synthetic.xlsx` was DRM-locked at first; Schneider sent an unlocked copy on 18 Nov 2026.
It holds **yearly** P10 / P50 / P90 selling prices (EUR ex-VAT, labelled "AI/ML", synthetic) for six brands, 2017–2026.
There are no monthly series and no 1F–4F configurations. The workbook's chart sheet draws an "SF6 marker" at 2025.5.
Eaton (Xiria) is the only SF6-free unit in the set.

In [ ]:
pb = dl.load_price_benchmark()
p50 = pb.pivot(index="year", columns="brand", values="p50")
yoy = (p50.pct_change() * 100).round(1)
print("Median price (P50), EUR:"); print(p50.loc[2024:].to_string())
print("\n2026 vs 2025 change, %:"); print(yoy.loc[2026].sort_values().to_string())

# Material cost vs selling price (RM6 = Schneider row)
rm6_cost = costs["RM6_NE_IQI_433520"].sum(axis=1)
se = pb[pb.brand == "Schneider Electric"].set_index("year")
cost_y = rm6_cost.groupby(rm6_cost.index.year).mean()
share = pd.DataFrame({"RM6 priced material, EUR (yearly mean)": cost_y.reindex(se.index).round(0),
                      "Schneider P50, EUR": se.p50,
                      "material % of price": (100 * cost_y.reindex(se.index) / se.p50).round(1)})
print(); print(share.to_string())
cu = costs["RM6_NE_IQI_433520"]["Copper"].iloc[-1]
print(f"\nLatest RM6 priced material {rm6_cost.iloc[-1]:.0f} EUR = {100*rm6_cost.iloc[-1]/se.p50[2026]:.1f}% of the 2026 P50;",
      f"teardown should-cost {bench.total.iloc[0]:.0f} EUR = {100*bench.total.iloc[0]/se.p50[2026]:.0f}%;",
      f"a 20% copper move = {0.2*cu:.0f} EUR = {100*0.2*cu/se.p50[2026]:.1f}% of price")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 3.9), gridspec_kw={"width_ratios": [1.35, 1]})
focus = {"Schneider Electric": PAL[0], "Eaton": PAL[2]}
for b in p50.columns:
    if b in focus: continue
    a1.plot(p50.index, p50[b], color=AXIS, lw=1.2)
    a1.annotate(b, (2026, p50.loc[2026, b]), xytext=(6, 0), textcoords="offset points", va="center", fontsize=7.5, color=MUTED)
a1.fill_between(se.index, se.p10, se.p90, color=PAL[0], alpha=0.12, lw=0)
for b, c in focus.items():
    a1.plot(p50.index, p50[b], color=c, lw=2)
    a1.annotate(f"{b}{' (SF6-free)' if b == 'Eaton' else ''}", (2026, p50.loc[2026, b]), xytext=(6, 0 if b == "Eaton" else -9),
                textcoords="offset points", va="center", fontsize=7.5, color=INK, fontweight="bold")
a1.axvline(2025.5, color=INK2, lw=1, ls=(0, (3, 3)))
a1.text(2025.45, a1.get_ylim()[1], "SF6 marker\n(in file)", ha="right", va="top", fontsize=7.5, color=INK2)
a1.set_xlim(2017, 2028.2); a1.set_xticks(range(2017, 2027, 1)); a1.set_xticklabels([f"{y%100:02d}" for y in range(2017, 2027)])
a1.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"€{v/1000:.0f}k"))
a1.set_title("Median selling price by brand (EUR ex-VAT); band = Schneider P10–P90")
idx = pd.DataFrame({"Schneider selling price (P50)": se.p50, "RM6 priced material cost": cost_y.reindex(se.index)})
idx = 100 * idx / idx.iloc[0]
for (col, c) in zip(idx.columns, [PAL[0], PAL[1]]):
    a2.plot(idx.index, idx[col], color=c, lw=2, marker="o", ms=4)
    a2.annotate(col, (idx.index[-1], idx[col].iloc[-1]), xytext=(6, 0), textcoords="offset points", va="center", fontsize=7.5, color=INK)
a2.axhline(100, color=AXIS, lw=0.8)
a2.set_xlim(2017, 2030.5); a2.set_xticks(range(2017, 2027, 1)); a2.set_xticklabels([f"{y%100:02d}" for y in range(2017, 2027)])
a2.set_title("Price vs material cost, 2017 = 100")
fig.tight_layout(); save(fig, "16_price_benchmark"); plt.show()

In [ ]:
ep = dl.load_epoxy()
# Data-entry error: May 2026 epichlorohydrin is ~10x below its neighbours, and the resin price that
# month is computed from it. Left in, it creates a spurious 0.87 ECH-resin correlation
# (June 2026 alone supplies ~91% of it). Exclude the May row from correlations and the fit.
typo = ep.index.strftime("%Y-%m") == "2026-05"
print("May 2026 row (typo):", ep.loc[typo, ["Epychlorhydrine", "Epoxy_resin"]].round(0).to_dict("records"))
r_raw = np.log(ep).diff()
ep = ep.loc[~typo].reindex(ep.index)  # keep the month as missing so monthly changes are not bridged across it
r = np.log(ep).diff()
print("Epoxy period:", ep.index.min().date(), "→", ep.index.max().date())
print("Correlation of monthly log-changes with epoxy resin (typo excluded | typo left in):")
print(pd.concat({"clean": r.corr()["Epoxy_resin"], "with_typo": r_raw.corr()["Epoxy_resin"]}, axis=1).round(2).to_string())
print("Benzene leading resin by k months (corr of log-changes):",
      {k: round(r.Benzene.shift(k).corr(r.Epoxy_resin), 2) for k in range(7)})
# implied linear cost build-up: epoxy ≈ a·benzene + b·acetone + c·ECH + other
fitdf = ep.dropna()
X = fitdf[["Benzene", "Acetone", "Epychlorhydrine", "Other"]]; y = fitdf["Epoxy_resin"]
coef, *_ = np.linalg.lstsq(np.c_[X.values, np.ones(len(X))], y.values, rcond=None)
fit = np.c_[X.values, np.ones(len(X))] @ coef
print("\nLinear recipe fit (t feedstock per t resin):", dict(zip(list(X.columns) + ["const"], coef.round(3))),
      f"| R² = {1 - ((y - fit)**2).sum() / ((y - y.mean())**2).sum():.3f}")

## 9 · Supplier financial health (PitchBook)

In [ ]:
fin = dl.load_supplier_financials()
latest = lambda item, st=None: (fin[(fin.item == item) & ((fin.statement == st) if st else True)]
                                .sort_values("year").groupby("supplier").agg(v=("value", "last"), y=("year", "last")))
# PitchBook labels revenue "Revenue" or "Total Revenue" depending on the filing template
fin.loc[(fin.statement == "Income Statement") & (fin.item == "Total Revenue"), "item"] = "Revenue"
# PitchBook's own 'Debt to Equity' ratio is 0 for every supplier (defect) -> recompute from balance sheet
print("PitchBook 'Debt to Equity' distinct values:", fin.loc[fin.item == "Debt to Equity", "value"].unique()[:5])
sup = pd.concat({
    "years": fin.groupby("supplier").year.agg(lambda y: f"{y.min()}–{y.max()}"),
    "n_datapoints": fin.groupby("supplier").size(),
    "revenue_kEUR": latest("Revenue", "Income Statement").v,
    "rev_year": latest("Revenue", "Income Statement").y,
    "EBITDA_kEUR": latest("EBITDA", "Income Statement").v,
    "EBITDA_margin": latest("EBITDA Margin").v,
    "debt_kEUR": latest("Total Debt", "Balance Sheet").v,
    "equity_kEUR": latest("Total Equity", "Balance Sheet").v,
    "current_ratio": latest("Current Ratio").v,
    "interest_cov": latest("Interest Coverage").v,
}, axis=1)
sup["debt_to_equity"] = sup.debt_kEUR / sup.equity_kEUR
sup["debt_to_EBITDA"] = sup.debt_kEUR / sup.EBITDA_kEUR.where(sup.EBITDA_kEUR > 0)
sup["flags"] = np.select(
    [sup.EBITDA_kEUR < 0, sup.current_ratio < 1, sup.debt_to_equity > 1.5, sup.revenue_kEUR.isna()],
    ["negative EBITDA", "current ratio < 1", "high leverage", "no P&L"], default="")
sup.sort_values("revenue_kEUR", ascending=False)

In [ ]:
rev = fin[(fin.statement == "Income Statement") & (fin.item == "Revenue")].pivot_table(index="year", columns="supplier", values="value")
rev = rev.loc[2015:]
fig, ax = plt.subplots(figsize=(11, 3.8))
for i, c in enumerate(rev.columns[:8]):
    s = rev[c].dropna(); ax.plot(s.index, s / 1000, marker="o", ms=4, color=PAL[i], label=c)
ax.set_title("Supplier revenue (EUR m) — note uneven coverage"); ax.legend(ncol=4, fontsize=7)
save(fig, "10_supplier_revenue"); plt.show()

## 10 · First forecasting baselines (h = 3 / 6 / 9 months)
Expanding-window backtest with one forecast origin per month from Jan-2020 onward.
The window covers both the 2021-22 shock and the calm that followed.
Any model we build must beat these three baselines:

- **naive:** last value
- **seasonal naive:** value 12 months before the target
- **drift:** random walk with the historical average drift

MAPE is reported in %.

In [ ]:
def backtest(s, h, start="2020-01-01"):
    s = s.dropna(); out = {"naive": [], "seasonal_naive": [], "drift": []}
    for t in s.index[(s.index >= start)]:
        i = s.index.get_loc(t)
        if i + h >= len(s) or i < 12: continue
        y_true, hist = s.iloc[i + h], s.iloc[:i + 1]
        drift = (hist.iloc[-1] - hist.iloc[0]) / (len(hist) - 1)
        preds = {"naive": hist.iloc[-1], "seasonal_naive": s.iloc[i + h - 12], "drift": hist.iloc[-1] + h * drift}
        for k, p in preds.items(): out[k].append(abs(p / y_true - 1))
    return {k: 100 * np.mean(v) for k, v in out.items()}

series = {**{c: rm[c] for c in ["D01", "D02", "S03", "S04", "S05", "S09", "S10", "S11"]},
          "CRU_CR_DE": steel.CRU_CR_DE.loc["2016-11":], "CRU_HDG_DE": steel.CRU_HDG_DE.loc["2016-11":], "CRU_SS304_DE": steel.CRU_SS304_DE.loc["2016-11":]}
bt = pd.DataFrame({(name, h): backtest(s, h) for name, s in series.items() for h in (3, 6, 9)}).T
bt.index.names = ["series", "h"]
bt.unstack("h").round(1)

In [ ]:
naive = bt["naive"].unstack("h")
fig, ax = plt.subplots(figsize=(11, 3.6))
x = np.arange(len(naive)); w = 0.26
for i, h in enumerate([3, 6, 9]):
    ax.bar(x + (i - 1) * w, naive[h], width=w - 0.03, color=PAL[i], label=f"h = {h}")
ax.set_xticks(x); ax.set_xticklabels(naive.index, rotation=0, fontsize=7.5)
ax.set_title("Naive (random-walk) backtest MAPE % by horizon — the bar to beat"); ax.legend()
save(fig, "11_baseline_mape"); plt.show()

## 11 · Take-aways for modelling
The headline findings are written up, with numbers, in [`DATA_OVERVIEW.md`](../DATA_OVERVIEW.md) §3.